# High-Volume ADE (DPT-3) → Snowflake — Demo Notebook

Parse + extract a batch of invoices with **LandingAI ADE DPT-3** (`client.v2.parse` /
`client.v2.extract`) and stream the structured results into **Snowflake** via staged
`COPY INTO`. The critical thing to watch is **step 5**: rows landing in Snowflake at a
high, continuous rate while documents are still being parsed.

Run cells top to bottom. Fill in `.env` (copy from `.env-sample`) and run
`snowflake_setup.sql` in your account first.

## 1. Configuration

Loads settings from `.env` / environment. The API key is never printed.

In [ ]:
from pprint import pprint
from config import Settings

S = Settings()
pprint(S.model_dump(exclude={"VISION_AGENT_API_KEY"}))

## 2. Discover input documents

In [ ]:
from run_demo import gather_files

files = gather_files(S.input_dir, S.file_exts)
print(f"{len(files)} document(s) found in {S.input_dir}")
files[:5]

## 3. Canary — parse + extract a single document

Validate the ADE path end-to-end before streaming the whole batch. DPT-3 splits this
into two calls: `client.v2.parse` then `client.v2.extract` (note the separate model
names — `dpt-3-pro-latest` for parse, `extract-latest` for extract).

In [ ]:
from ade_client import build_client, parse_and_extract
from invoice_schema import InvoiceExtractionSchema

client = build_client(S)
parse_result, extract_result = parse_and_extract(client, files[0], InvoiceExtractionSchema, S)

n_blocks = sum(len(p.children or []) for p in (parse_result.structure.children or []))
print("pages:", parse_result.metadata.page_count, "| blocks:", n_blocks,
      "| 206:", extract_result.schema_violation_error)
extract_result.extraction

### Build the Snowflake rows from that document

`rows_from_doc` turns the parse `structure` tree + extraction into four row sets:
header, line items, parsed blocks, and full markdown.

In [ ]:
from datetime import datetime, timezone
from row_builder import rows_from_doc

main, lines, blocks, md_record, uid = rows_from_doc(
    fp=files[0], parse_result=parse_result, extract_result=extract_result,
    run_id="CANARY", sent_at=datetime.now(timezone.utc), sdk_version="notebook",
)
print("header:", {k: main[k] for k in ('invoice_number','supplier_name','total_due','currency')})
print("line items:", len(lines), "| blocks:", len(blocks))
blocks[:3]

## 4. Create Snowflake stages & file formats

Idempotent — safe to re-run. (Tables, role, and grants come from `snowflake_setup.sql`,
which you run once in your account.)

In [ ]:
from sf_loader import ensure_formats_and_stages

ensure_formats_and_stages(S)
print("ingest stage + file formats ready")

## 5. Stream the batch into Snowflake — watch the rate

This is the money shot. A thread pool parses+extracts many documents at once and each
one's rows are staged and `COPY`ed the moment it finishes, so rows land continuously.

**Tip:** open a Snowsight worksheet and run `SELECT COUNT(*) FROM INVOICES_MAIN;`
repeatedly while this cell runs — the counts climb in real time.

To simulate volume from the few sample invoices, uncomment the `replicate` line
(each copy is parsed independently and **consumes ADE credits**).

In [ ]:
from run_demo import replicate
from pipeline import run_streaming

batch = files
# batch = replicate(files, 250)   # <-- uncomment to simulate high volume

metrics = run_streaming(batch, InvoiceExtractionSchema, S)
print(metrics.summary())

## 6. Verify what landed

In [ ]:
from run_demo import verify_counts

verify_counts(S)

## 7. Query the results

From here it's just SQL — the extracted fields are query-ready and grounded.

In [ ]:
from sf_loader import sfcursor, fq_table

query = f"""
SELECT m.supplier_name, m.invoice_number, m.total_due, COUNT(li.line_index) AS lines
FROM {fq_table(S, S.table_main)} m
LEFT JOIN {fq_table(S, S.table_lines)} li USING (invoice_uuid)
GROUP BY 1, 2, 3
ORDER BY m.total_due DESC
LIMIT 10
"""
with sfcursor(settings=S) as cur:
    cur.execute(query)
    for row in cur.fetchall():
        print(row)

## Next steps

- Swap `invoice_schema.py` for your own Pydantic schema (and update `COLS_*` + the DDL).
- Raise `MAX_WORKERS` and warehouse size to push the rate higher.
- Feed `PARSED_BLOCKS` / `MARKDOWN` into a RAG index.

Docs: https://docs.landing.ai/dpt3/ade-python  ·  Support: https://docs.landing.ai/ade/ade-support